# Chatbot de Joyería — Construcción Correcta
### Proyecto I: Introducción a LLMs — Facultad de Ciencias, UNAM — Semestre 2026-2

---

> **¿Por qué un notebook separado?** El baseline anterior usó `flan-t5-base`, un modelo seq2seq que no fue diseñado para seguir instrucciones conversacionales — producía respuestas en varios idiomas o sin sentido. Este notebook construye el chatbot correctamente usando un modelo **instruct**.

### ¿Qué es un modelo instruct?

Un modelo base (como GPT-2 o Qwen base) solo sabe completar texto. Un modelo **instruct** pasó por una segunda fase de entrenamiento — RLHF o SFT — que le enseña a:
- Seguir instrucciones del sistema
- Mantener el rol asignado
- Responder en el idioma del usuario
- No alucinar fuera del contexto dado

```
Modelo base:    "El gato..." → completa con lo más probable
Modelo instruct: [system: eres un asistente...] [user: hola] → responde apropiadamente
```

**Mapa de esta sesión:**
```
Parte 1: Por qué falló el baseline — diagnóstico          (10 min)
Parte 2: Modelos instruct y chat templates                (15 min)
Parte 3: Chatbot funcional con Qwen2.5-Instruct           (20 min)
Parte 4: Evaluar y documentar el baseline correcto        (15 min)
```

**Infraestructura:** Colab T4 GPU (Entorno de ejecución → Cambiar tipo → T4 GPU)

In [ ]:
!pip install transformers torch accelerate bitsandbytes --quiet

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    memoria_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM disponible: {memoria_gb:.1f} GB")
else:
    print("⚠ Sin GPU — activa T4 en Entorno de ejecución → Cambiar tipo de entorno")

---
## Parte 1: Diagnóstico — Por qué Falló el Baseline

El problema con `flan-t5-base` no fue el prompt sino el modelo. Hay tres razones:

**1. Es seq2seq, no causal.** Flan-T5 fue entrenado para mapear una secuencia de entrada a una de salida (como traducción o resumen), no para conversación libre.

**2. No tiene chat template.** Los modelos conversacionales modernos esperan un formato específico con roles (`system`, `user`, `assistant`). Sin ese formato, el modelo no sabe quién habla ni qué se espera de él.

**3. El español no era su fortaleza.** Flan-T5-base fue entrenado principalmente en inglés. Con instrucciones mixtas (inglés + español) el modelo se confunde sobre en qué idioma responder.

### La solución: chat template + modelo instruct

In [ ]:
# El chat template: cómo se estructuran los mensajes para un modelo instruct
# Cada modelo tiene su propio formato — HuggingFace lo maneja automáticamente

# Ejemplo con Qwen2.5 (el modelo que usaremos)
# El tokenizador convierte esta lista de mensajes al formato que espera el modelo:

mensajes_ejemplo = [
    {
        "role": "system",
        "content": "Eres un asistente de ventas para una tienda de joyería artesanal."
    },
    {
        "role": "user",
        "content": "¿Qué aretes tienen disponibles?"
    }
]

# El tokenizador aplica el template y produce algo así:
print("Estructura de mensajes (antes del template):")
for msg in mensajes_ejemplo:
    print(f"  [{msg['role']}]: {msg['content']}")

print()
print("Después de aplicar el chat template de Qwen, el texto se ve así:")
print("  <|im_start|>system")
print("  Eres un asistente de ventas para una tienda de joyería artesanal.<|im_end|>")
print("  <|im_start|>user")
print("  ¿Qué aretes tienen disponibles?<|im_end|>")
print("  <|im_start|>assistant")
print("  [el modelo genera aquí]")
print()
print("Estos tokens especiales (<|im_start|>, <|im_end|>) le dicen al modelo")
print("cuándo empieza y termina cada turno de la conversación.")
print("Sin ellos, el modelo no puede distinguir roles ni estructura.")

---
## Parte 2: Selección de Modelo

Para un chatbot en español que corra en Colab gratuito (T4, ~15 GB VRAM), los candidatos son:

| Modelo | Params | VRAM | Español | Instrucciones |
|:---|:---:|:---:|:---:|:---:|
| `Qwen2.5-1.5B-Instruct` | 1.5B | ~3 GB | ★★★★ | ★★★★★ |
| `Qwen2.5-0.5B-Instruct` | 0.5B | ~1 GB | ★★★ | ★★★★ |
| `TinyLlama-1.1B-Chat` | 1.1B | ~2 GB | ★★ | ★★★ |
| `microsoft/Phi-3-mini-4k-instruct` | 3.8B | ~7 GB | ★★★ | ★★★★★ |
| `mistralai/Mistral-7B-Instruct-v0.3` | 7B | ~14 GB | ★★★★ | ★★★★★ |

**Elección: `Qwen2.5-1.5B-Instruct`**
- Excelente manejo del español para su tamaño
- Corre sin cuantización en T4 (~3 GB VRAM)
- Sigue instrucciones del sistema de forma confiable
- No requiere aprobación de Meta (a diferencia de LLaMA)
- Misma familia que usarán en fine-tuning (Qwen2.5 0.6B–4B)

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Cargando {MODEL_ID}...")
print("(Primera vez: descarga ~3 GB — toma 2-3 minutos)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Cuantización opcional: si la VRAM es limitada, activar 4-bit
# En T4 con 15GB no es necesario para 1.5B, pero lo mostramos como patrón
usar_4bit = False  # cambiar a True si hay problemas de memoria

if usar_4bit:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map="auto"
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
        device_map="auto" if device == 'cuda' else None
    )
    if device == 'cpu':
        model = model.to('cpu')

model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"\nModelo cargado.")
print(f"Parámetros: {total_params/1e9:.2f}B")
if device == 'cuda':
    vram_usada = torch.cuda.memory_allocated() / 1e9
    print(f"VRAM usada: {vram_usada:.2f} GB")

In [ ]:
# Verificar que el modelo funciona correctamente en español
# ANTES de integrar el catálogo

def generar_respuesta(mensajes, max_tokens=256, temperatura=0.7):
    """
    Genera una respuesta dado un historial de mensajes.
    
    mensajes: lista de dicts con 'role' y 'content'
    El tokenizador aplica el chat template automáticamente.
    """
    # Aplicar chat template — convierte la lista de mensajes al formato del modelo
    texto = tokenizer.apply_chat_template(
        mensajes,
        tokenize=False,
        add_generation_prompt=True  # agrega el prompt para que el modelo empiece a generar
    )
    
    inputs = tokenizer(texto, return_tensors='pt').to(model.device)
    n_tokens_entrada = inputs['input_ids'].shape[1]
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperatura,
            do_sample=temperatura > 0,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decodificar solo los tokens NUEVOS (no el prompt)
    tokens_nuevos = outputs[0][n_tokens_entrada:]
    respuesta = tokenizer.decode(tokens_nuevos, skip_special_tokens=True)
    return respuesta.strip()


# Prueba básica
print("Prueba básica — ¿funciona el español?")
mensajes_prueba = [
    {"role": "system", "content": "Eres un asistente útil. Responde siempre en español."},
    {"role": "user", "content": "¿Cuál es la capital de México y qué es lo más famoso de esa ciudad?"}
]
respuesta = generar_respuesta(mensajes_prueba)
print(f"\nRespuesta: {respuesta}")
print()
print("Si la respuesta está en español y tiene sentido, el modelo funciona correctamente.")

---
## Parte 3: Chatbot Funcional — Taller Luna Plata

Ahora construimos el chatbot completo. La arquitectura tiene tres capas:

```
┌─────────────────────────────────────────────────────┐
│  System prompt                                      │
│  → Define el rol, el tono, y las reglas del chatbot │
│  → Incluye el catálogo completo de productos        │
├─────────────────────────────────────────────────────┤
│  Historial de conversación                          │
│  → Mensajes anteriores del usuario y del chatbot    │
│  → Permite conversación multi-turno                 │
├─────────────────────────────────────────────────────┤
│  Mensaje actual del usuario                         │
│  → La pregunta de este turno                        │
└─────────────────────────────────────────────────────┘
               ↓
         Modelo genera
               ↓
         Respuesta del asistente
```

Esto es exactamente el patrón RAG en su forma más simple: el catálogo actúa como la base de conocimiento recuperada y se inyecta en el contexto.

In [ ]:
# Catálogo de la tienda — en el proyecto real viene de una base de datos
CATALOGO = """
CATÁLOGO DE JOYERÍA ARTESANAL — TALLER LUNA PLATA

ARETES:
- Aretes de plata 925 con turquesa natural. Precio: $650 MXN. Disponibles en azul y verde.
- Aretes de cobre con baño de oro y piedra ojo de tigre. Precio: $480 MXN. Stock limitado (3 pares).
- Aretes minimalistas de plata lisa, forma de luna. Precio: $320 MXN. Siempre disponibles.
- Aretes largos de plata con cuentas de obsidiana. Precio: $580 MXN. Disponibles.

COLLARES:
- Collar de plata con dije de obsidiana tallada a mano. Precio: $1,200 MXN. Cada pieza es única.
- Collar de cuarzo rosa en cadena de plata 925. Precio: $980 MXN. Disponible.
- Gargantilla de cobre trenzado. Precio: $550 MXN. Disponible en 3 grosores (delgado, mediano, grueso).
- Collar choker de plata con piedra labradorita. Precio: $1,100 MXN. Stock limitado (2 piezas).

PULSERAS:
- Pulsera de plata con charms artesanales (colibrí, luna, sol). Precio: $750 MXN. Personalizable.
- Brazalete de cobre martillado. Precio: $420 MXN. Talla única ajustable.
- Pulsera de plata con piedra de luna (moonstone). Precio: $890 MXN. Disponible.

INFORMACIÓN GENERAL:
- Envíos a toda la República Mexicana. Costo: $120 MXN (GRATIS en compras mayores a $1,500 MXN).
- Tiempo de entrega: 3-5 días hábiles en CDMX, 5-7 días en el resto del país.
- Métodos de pago aceptados: tarjeta de crédito/débito, transferencia bancaria (SPEI), PayPal.
- Pedidos personalizados disponibles con tiempo adicional de 7-10 días hábiles.
- Garantía: 6 meses en piezas de plata 925 contra defectos de fabricación.
- Política de devolución: 30 días en producto sin uso y en empaque original.
- No se trabaja con oro macizo ni metales preciosos distintos a plata 925 y cobre.
- Contacto: Instagram @tallerlunaplata | Email: contacto@tallerlunaplata.com | WhatsApp: 55-1234-5678
"""

# System prompt: define el rol y las reglas del chatbot
SYSTEM_PROMPT = f"""Eres Luna, la asistente virtual del Taller Luna Plata, una tienda de joyería artesanal mexicana.

Tu personalidad:
- Eres amable, cálida y apasionada por la joyería artesanal
- Conoces profundamente cada pieza del catálogo
- Siempre respondes en español, sin importar el idioma del cliente
- Eres honesta: si algo no está en el catálogo, lo dices claramente

Tus reglas:
- Solo ofreces productos y servicios que aparecen en el catálogo
- No inventas productos, precios ni disponibilidades
- Si el cliente pregunta algo que no sabes, ofreces el contacto de la tienda
- Mantienes respuestas concisas (máximo 3-4 oraciones por respuesta)

CATÁLOGO ACTUAL:
{CATALOGO}"""

print(f"System prompt listo.")
print(f"Longitud del system prompt: {len(SYSTEM_PROMPT)} caracteres")
print(f"Tokens aproximados: ~{len(SYSTEM_PROMPT.split())*1.3:.0f}")

In [ ]:
class ChatbotJoyeria:
    """
    Chatbot conversacional con memoria de contexto.
    
    El historial se acumula en self.mensajes.
    En cada turno, el modelo ve toda la conversación anterior.
    """
    
    def __init__(self, system_prompt, max_historial=10):
        self.system_prompt = system_prompt
        self.max_historial = max_historial  # limitar contexto para no exceder VRAM
        self.mensajes = []  # historial de la conversación
    
    def chat(self, pregunta_usuario, verbose=False):
        # Agregar el mensaje del usuario al historial
        self.mensajes.append({"role": "user", "content": pregunta_usuario})
        
        # Construir el contexto completo: system + historial reciente
        # Truncar historial si es muy largo (ventana de contexto limitada)
        historial_reciente = self.mensajes[-self.max_historial:]
        contexto = [{"role": "system", "content": self.system_prompt}] + historial_reciente
        
        if verbose:
            n_turnos = len(self.mensajes)
            print(f"[Turno {n_turnos} | Mensajes en contexto: {len(contexto)}]")
        
        # Generar respuesta
        respuesta = generar_respuesta(contexto, max_tokens=200, temperatura=0.7)
        
        # Agregar la respuesta al historial
        self.mensajes.append({"role": "assistant", "content": respuesta})
        
        return respuesta
    
    def reiniciar(self):
        """Nueva conversación — borra el historial."""
        self.mensajes = []
        print("Conversación reiniciada.")
    
    def mostrar_historial(self):
        for msg in self.mensajes:
            rol = "👤" if msg['role'] == 'user' else "🤖"
            print(f"{rol} {msg['content']}")
            print()


# Instanciar el chatbot
luna = ChatbotJoyeria(system_prompt=SYSTEM_PROMPT)
print("Chatbot Luna inicializado.")

In [ ]:
# Batería de pruebas — casos de uso principales
# Cada pregunta prueba un aspecto diferente del chatbot

preguntas_prueba = [
    # Información de producto
    ("¿Qué aretes tienen disponibles y cuánto cuestan?",         "info de producto"),
    ("¿Cuánto cuesta el collar de obsidiana?",                   "precio específico"),
    ("¿De qué materiales está hecha la gargantilla de cobre?",   "detalle de material"),
    # Logística
    ("¿Hacen envíos a Guadalajara? ¿Cuánto cuesta?",             "logística"),
    ("¿Cuánto tiempo tarda en llegar un pedido a Monterrey?",    "tiempo de entrega"),
    # Pagos y políticas
    ("¿Puedo pagar con transferencia bancaria?",                 "método de pago"),
    ("¿Qué pasa si el producto llega dañado?",                   "garantía/devolución"),
    # Personalización
    ("¿Puedo pedir una pulsera con el nombre de mi pareja?",     "personalización"),
    # Fuera del catálogo — el chatbot debe reconocerlo
    ("¿Tienen anillos de compromiso en oro?",                    "fuera de catálogo"),
    ("¿Tienen descuentos si compro varios artículos?",           "fuera de catálogo"),
]

print("=" * 65)
print("BATERÍA DE PRUEBAS — Chatbot Taller Luna Plata")
print("=" * 65)

luna.reiniciar()
resultados_prueba = []

for pregunta, categoria in preguntas_prueba:
    print(f"\n[{categoria.upper()}]")
    print(f"👤 {pregunta}")
    respuesta = luna.chat(pregunta)
    print(f"🤖 {respuesta}")
    resultados_prueba.append((pregunta, categoria, respuesta))

In [ ]:
# Prueba multi-turno: el chatbot debe recordar el contexto de la conversación

print("=" * 65)
print("PRUEBA MULTI-TURNO — ¿Recuerda el contexto?")
print("=" * 65)
print()

luna.reiniciar()

conversacion = [
    "Hola, estoy buscando un regalo para el cumpleaños de mi mamá.",
    "Le gustan los collares. ¿Cuál me recomiendas?",
    "¿Cuánto cuesta ese?",                         # referencia al collar anterior
    "¿Y el envío está incluido si lo compro?",      # referencia al precio anterior
    "Perfecto. ¿Cómo lo puedo pagar?",
]

for turno, mensaje in enumerate(conversacion, 1):
    print(f"Turno {turno}:")
    print(f"👤 {mensaje}")
    respuesta = luna.chat(mensaje, verbose=True)
    print(f"🤖 {respuesta}")
    print()

print("Si el chatbot respondió coherentemente en los turnos 3-5 (usando contexto")
print("de los mensajes anteriores), la memoria de conversación funciona.")

In [ ]:
# Modo interactivo — para probar el chatbot en tiempo real en clase
# Ejecutar esta celda y escribir preguntas

print("=" * 50)
print("MODO INTERACTIVO — Chatbot Luna")
print("Escribe 'salir' para terminar, 'nuevo' para nueva conversación")
print("=" * 50)

luna.reiniciar()

while True:
    try:
        entrada = input("\n👤 Tú: ").strip()
    except EOFError:
        break
    
    if not entrada:
        continue
    if entrada.lower() == 'salir':
        print("Conversación terminada.")
        break
    if entrada.lower() == 'nuevo':
        luna.reiniciar()
        continue
    
    respuesta = luna.chat(entrada)
    print(f"🤖 Luna: {respuesta}")

---
## Parte 4: Evaluar y Documentar el Baseline Correcto

Ahora que el chatbot funciona, necesitamos medir su desempeño de forma reproducible. Para chatbots hay dos tipos de evaluación:

**Evaluación automática** — rápida pero imperfecta:
- Verificar si la respuesta menciona precios correctos (extracción de entidades)
- Detectar alucinaciones (información que no está en el catálogo)

**Evaluación humana** — costosa pero confiable:
- Rúbrica con criterios claros
- Al menos 20 preguntas representativas
- Mismo evaluador antes y después del fine-tuning

In [ ]:
# Evaluación automática: detección de alucinaciones
# Verificar si el chatbot inventó información que no está en el catálogo

import re

# Precios que SÍ existen en el catálogo
precios_reales = {320, 420, 480, 550, 580, 650, 750, 890, 980, 1100, 1200}

def extraer_precios_mencionados(texto):
    """Extrae precios mencionados en la respuesta."""
    # Buscar patrones: $650, $1,200, 650 pesos, 1200 MXN
    patrones = re.findall(r'\$?([\d,]+)(?:\s*(?:MXN|pesos))?', texto)
    precios = set()
    for p in patrones:
        try:
            precio = int(p.replace(',', ''))
            if 100 <= precio <= 5000:  # rango razonable para joyería
                precios.add(precio)
        except:
            pass
    return precios

def detectar_alucinaciones_precio(respuesta):
    """Detecta si el chatbot mencionó precios que no existen en el catálogo."""
    precios_mencionados = extraer_precios_mencionados(respuesta)
    alucinados = precios_mencionados - precios_reales
    correctos = precios_mencionados & precios_reales
    return correctos, alucinados


print("Análisis de alucinaciones en precios")
print("=" * 55)

n_correctos = 0
n_alucinaciones = 0

for pregunta, categoria, respuesta in resultados_prueba:
    correctos, alucinados = detectar_alucinaciones_precio(respuesta)
    if correctos or alucinados:
        estado = "✓" if not alucinados else "⚠ ALUCINACIÓN"
        print(f"\nPregunta: {pregunta[:55]}")
        if correctos:
            print(f"  Precios correctos mencionados: {correctos}")
        if alucinados:
            print(f"  {estado}: Precios inventados: {alucinados}")
            n_alucinaciones += 1
        else:
            n_correctos += 1

total_con_precios = n_correctos + n_alucinaciones
if total_con_precios > 0:
    print(f"\nTasa de alucinación en precios: {n_alucinaciones/total_con_precios*100:.0f}%")
print()
print("Nota: esta es una métrica parcial. La evaluación humana con rúbrica")
print("es indispensable para medir calidad real.")

In [ ]:
# Rúbrica de evaluación humana
# Llenar para cada respuesta en la batería de 10 preguntas

print("Rúbrica de evaluación — Chatbot Taller Luna Plata")
print("=" * 60)

criterios = [
    ("Precisión factual",
     "Los datos del catálogo (precios, disponibilidad, materiales) son correctos",
     3, ["0: dato incorrecto", "1: parcialmente correcto", "2: correcto con omisiones", "3: completamente correcto"]),
    
    ("Completitud",
     "La respuesta cubre todos los aspectos preguntados",
     2, ["0: ignora parte de la pregunta", "1: responde parcialmente", "2: responde todo"]),
    
    ("Tono y rol",
     "Mantiene el rol de Luna, tono cálido, en español",
     2, ["0: fuera de rol o idioma incorrecto", "1: parcialmente en rol", "2: en rol completamente"]),
    
    ("Manejo de límites",
     "Reconoce honestamente cuando algo no está en el catálogo",
     3, ["0: inventa información", "1: evade la pregunta", "2: dice que no sabe", "3: dice que no sabe Y ofrece alternativa"]),
]

puntuacion_max = sum(c[2] for c in criterios)
print(f"Puntuación máxima: {puntuacion_max} puntos por respuesta")
print()

for nombre, descripcion, max_pts, niveles in criterios:
    print(f"[{nombre}] — máx {max_pts} pts")
    print(f"  {descripcion}")
    for nivel in niveles:
        print(f"    {nivel}")
    print()

print("─" * 60)
print("Proceso de evaluación:")
print("  1. Evalúa las 10 respuestas de la batería de pruebas")
print("  2. Suma puntos por respuesta y calcula el promedio")
print(f"  3. Expresa como porcentaje: (total / {puntuacion_max*10}) × 100")
print("  4. Este es tu score de baseline — regístralo en tu tarjeta de experimento")

In [ ]:
# Tarjeta de experimento — baseline del chatbot
import json
from datetime import datetime

tarjeta_baseline = {
    "fecha": datetime.now().strftime("%Y-%m-%d"),
    "experimento": "baseline-v1",
    "proyecto": "Chatbot joyería artesanal — Taller Luna Plata",
    "modelo": "Qwen/Qwen2.5-1.5B-Instruct",
    "estrategia": "prompt engineering + catálogo en contexto (RAG simple)",
    "parametros_generacion": {
        "max_new_tokens": 200,
        "temperature": 0.7,
        "top_p": 0.9,
        "repetition_penalty": 1.1
    },
    "metricas": {
        "score_rubrica": "[COMPLETAR]",   # promedio de las 10 preguntas
        "tasa_alucinacion": "[COMPLETAR]", # % respuestas con info incorrecta
        "idioma_correcto": "[COMPLETAR]",  # % respuestas en español
    },
    "observaciones": [
        "[COMPLETAR: qué funcionó bien]",
        "[COMPLETAR: qué falló]",
        "[COMPLETAR: casos borde identificados]",
    ],
    "siguiente_experimento": "fine-tuning con LoRA sobre conversaciones de ejemplo"
}

print("Tarjeta de experimento — guardar en tu repositorio:")
print()
print(json.dumps(tarjeta_baseline, ensure_ascii=False, indent=2))
print()
print("Guardar como: experimentos/baseline_chatbot_v1.json")

---
## Siguiente Paso

Este baseline usa el catálogo directamente en el prompt — funciona, pero tiene dos limitaciones:

1. **Contexto limitado:** Con catálogos grandes (cientos de productos), no cabe en el contexto del modelo
2. **Sin personalidad propia:** El modelo responde bien pero no tiene el estilo específico de la tienda

Las dos rutas de mejora son:

```
Ruta A — RAG completo (el TA cubre esto):
  Catálogo → embeddings semánticos → búsqueda por similitud
  → solo se inyecta la parte relevante del catálogo
  → escala a catálogos grandes sin límite de contexto

Ruta B — Fine-tuning con LoRA (Fase 2):
  Conversaciones de ejemplo → LoRA sobre Qwen2.5
  → el modelo aprende el estilo de la tienda
  → respuestas más naturales y consistentes
```

El MVP completo combina ambas: RAG para el catálogo + fine-tuning para el estilo.